John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

Adapted from David Logue's code

### Import Packages

In [1]:
# Import packages

# library(geosphere)
library(tidyverse)
# library(dagitty)
# library(brms)
# library(forestplot)
# library(lqmm)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'tibble' was built under R version 4.4.3"
Warning message:
"package 'tidyr' was built under R version 4.4.3"
Warning message:
"package 'readr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'forcats' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package 

### Load and Trim Dataset

In [2]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/megazord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

New names:
• `` -> `...1`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 12799 Columns: 127
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (14): ID, recordist, Begin.Path, Location_Number, Comments, ST1, Data_...
dbl  (104): ...1, X.2, X.1, selec, start, end, File.Offset..s., SNR.NIST.Qui...
lgl    (2): RepeatST, GreaterThanT
dttm   (1): DateTime
date   (1): date
time   (5): click_clock, start_hms, click_clock_hms, clock_time, sunrise

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [ ]:
# Get all unique focal male IDs (excluding NAs)
focal_ids <- unique(na.omit(data$ID))
print(focal_ids)

# Initialize an empty list to store the output dataframes
focal_outputs <- list()

# Just focusing on one focal male for now (central male YKY)
focal_id <- "YKY"
# Get rows in which ID == focal_id
focals_data <- filter(data, ID == focal_id)

# Get unique set of USTs for the focal (including NA values)
focal_repertoire <- unique(focals_data$UST)

# Find all rows that have a UST in focal's repertoire, not including NA values
focal_ind <- data %>% filter(UST %in% focal_repertoire & !is.na(UST))
# Group these rows by date, create columns to show focal's next UST
focal_ind <- focal_ind %>%
    group_by(date) %>% # Group by date so we're not comparing "last song" across days
    mutate(focal_next_UST = if_else(ID == focal_id, UST, NA)) %>%
    fill(focal_next_UST, .direction = "up")

# Create column focal_matches : binary variable (is there a song type match?)
# If the focal bird is the singer, it cannot match itself, return 0
# If the focal bird is not the singer, does the song type match the focal's next song? (within day)
focal_ind <- focal_ind %>%
    mutate(focal_matches = if_else(ID == focal_id, NA,
                                    if_else(focal_next_UST == UST, 1, 0)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


### Calculate Distances for Each Matching Opportunity